# Cryptocurrency Forecasting Pipeline (BTC/ETH)
Mount Google Drive, lưu CSV từng bước, và lưu kết quả ra CSV.

In [ ]:
# @title # Mount Data
from google.colab import drive
import os

# Ensure the mount point is empty before mounting
if os.path.exists('/content/drive'):
    # Only remove if it's a directory and not already a mount point
    if os.path.isdir('/content/drive') and not os.path.ismount('/content/drive'):
        print("Clearing existing /content/drive directory...")
        for root, dirs, files in os.walk('/content/drive', topdown=False):
            for name in files:
                os.remove(os.path.join(root, name))
            for name in dirs:
                os.rmdir(os.path.join(root, name))
    elif os.path.isfile('/content/drive'):
        print("Removing file at /content/drive...")
        os.remove('/content/drive')

# Attempt to create the directory if it doesn't exist (it should be empty now)
os.makedirs('/content/drive', exist_ok=True)

drive.mount('/content/drive', force_remount=True)

DATA_DIR = "/content/drive/MyDrive/crypto_pipeline_data"
os.makedirs(DATA_DIR, exist_ok=True)
print("Data directory:", DATA_DIR)

In [ ]:
import os
import io
import requests
import numpy as np
import pandas as pd
from typing import List

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Bidirectional, LSTM

BITINFOCHARTS_BASE = "https://bitinfocharts.com/comparison/"

BITINFO_METRICS = {
    "transactions": "transactions",
    "blocksize": "blocksize",
    "sentbyaddress": "sentbyaddress",
    "difficulty": "difficulty",
    "mining_profitability": "mining_profitability",
    "sentcoin": "sentcoin",
    "fee_to_reward": "fee_to_reward",
    "median_tx_fee": "median_tx_fee",
    "average_tx_fee": "average_tx_fee",
    "blocktime": "blocktime",
    "hashrate": "hashrate",
    "median_tx_value": "median_tx_value",
    "activeaddresses": "activeaddresses",
    "top100cap": "top100cap",
    "average_tx_value": "average_tx_value",
}

WINDOWS = [3, 7, 14, 30, 90]
HORIZONS = [1, 7, 14, 30, 60, 90]

INTERVALS = {
    "interval1": ("2013-04-01", "2016-04-01"),
    "interval2": ("2013-04-01", "2017-04-01"),
    "interval3": ("2013-04-01", "2019-12-31"),
}
CUSUM_TEST = ("2020-01-01", "2022-01-01")

def save_csv(df, name):
    path = os.path.join(DATA_DIR, name)
    df.to_csv(path, index=False)
    print(f"Saved: {path}")
    return path

def load_csv(name):
    path = os.path.join(DATA_DIR, name)
    df = pd.read_csv(path)
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
    print(f"Loaded: {path}")
    return df

In [ ]:
def fetch_bitinfocharts_metric(metric: str, coin: str) -> pd.DataFrame:
    url = f"{BITINFOCHARTS_BASE}{metric}-{coin}.csv"
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text), header=None, names=["date", metric])
    df = df[df["date"].str.match(r"\d{4}-\d{2}-\d{2}")]
    df["date"] = pd.to_datetime(df["date"])
    df[metric] = pd.to_numeric(df[metric], errors="coerce")
    return df

def fetch_all_bitinfocharts(coin: str) -> pd.DataFrame:
    dfs = []
    for name, metric in BITINFO_METRICS.items():
        df = fetch_bitinfocharts_metric(metric, coin)
        df = df.rename(columns={metric: name})
        dfs.append(df)
    merged = dfs[0]
    for df in dfs[1:]:
        merged = merged.merge(df, on="date", how="outer")
    merged = merged.sort_values("date")
    return merged

def fetch_ohlc_coingecko(coin_id: str) -> pd.DataFrame:
    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart?vs_currency=usd&days=max"
    r = requests.get(url)
    r.raise_for_status()
    data = r.json()
    prices = pd.DataFrame(data["prices"], columns=["ts", "close"])
    prices["date"] = pd.to_datetime(prices["ts"], unit="ms").dt.date
    daily = prices.groupby("date")["close"].agg(["first", "max", "min", "last"]).reset_index()
    daily.columns = ["date", "open", "high", "low", "close"]
    daily["date"] = pd.to_datetime(daily["date"])
    return daily

In [ ]:
def SMA(series, window): return series.rolling(window).mean()
def EMA(series, window): return series.ewm(span=window, adjust=False).mean()
def WMA(series, window):
    weights = np.arange(1, window + 1)
    return series.rolling(window).apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)
def RSI(series, window=14):
    delta = series.diff()
    gain = np.where(delta > 0, delta, 0)
    loss = np.where(delta < 0, -delta, 0)
    avg_gain = pd.Series(gain).rolling(window).mean()
    avg_loss = pd.Series(loss).rolling(window).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    return 100 - (100 / (1 + rs))
def STD(series, window): return series.rolling(window).std()
def VAR(series, window): return series.rolling(window).var()
def TRIX(series, window):
    ema1 = series.ewm(span=window, adjust=False).mean()
    ema2 = ema1.ewm(span=window, adjust=False).mean()
    ema3 = ema2.ewm(span=window, adjust=False).mean()
    return ema3.pct_change() * 100
def ROC(series, window): return series.pct_change(periods=window) * 100

def add_indicators(df: pd.DataFrame, base_cols: List[str]) -> pd.DataFrame:
    for col in base_cols:
        for w in WINDOWS:
            df[f"{col}_SMA_{w}"] = SMA(df[col], w)
            df[f"{col}_EMA_{w}"] = EMA(df[col], w)
            df[f"{col}_RSI_{w}"] = RSI(df[col], w)
            df[f"{col}_WMA_{w}"] = WMA(df[col], w)
            df[f"{col}_STD_{w}"] = STD(df[col], w)
            df[f"{col}_VAR_{w}"] = VAR(df[col], w)
            df[f"{col}_TRIX_{w}"] = TRIX(df[col], w)
            df[f"{col}_ROC_{w}"] = ROC(df[col], w)
    return df

In [ ]:
def preprocess_split(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col].values

    imputer = SimpleImputer(strategy="most_frequent")
    X_imputed = imputer.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_imputed, y, test_size=0.33, random_state=42, shuffle=False
    )

    scaler = MinMaxScaler(feature_range=(0, 1))
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, y_train, y_test, X.columns

def select_features_mdi(X_train, y_train, feature_names):
    rf = RandomForestRegressor(bootstrap=False, random_state=42)
    rf.fit(X_train, y_train)

    result = permutation_importance(rf, X_train, y_train, random_state=42)
    mean = result.importances_mean
    std = result.importances_std

    keep_idx = []
    for i in range(len(feature_names)):
        if mean[i] - (2 * std[i]) > 0:
            keep_idx.append(i)
    return keep_idx

In [ ]:
def build_bilstm(input_shape):
    model = Sequential()
    model.add(Bidirectional(LSTM(128, activation="relu"), input_shape=input_shape))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(loss="mse", optimizer="adam")
    return model

def make_sliding_window(X, y, window=1):
    Xs, ys = [], []
    for i in range(len(X) - window):
        Xs.append(X[i:i+window])
        ys.append(y[i+window])
    return np.array(Xs), np.array(ys)

def CUSUM_Control_Chart(predictions, actuals, model, X_recent, y_recent):
    preds = np.array(predictions)
    n = len(preds)
    variance = np.var(preds)
    std = np.sqrt(variance)
    std = std / n

    upper = std * 3
    lower = -std * 3

    cumsum = 0
    for i in range(len(preds)):
        deviation = preds[i] - actuals[i]
        cumsum += deviation
        if cumsum > upper or cumsum < lower:
            print("CUSUM WARNING: bias detected, retraining on recent data...")
            model.fit(X_recent, y_recent, epochs=10, batch_size=32, verbose=0)
            break

def report_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mae = np.mean(np.abs(y_true - y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, mape, r2

In [ ]:
def run_pipeline(coin: str, coin_id: str):
    # Step 1: Fetch + Save raw
    df_metrics = fetch_all_bitinfocharts(coin)
    df_ohlc = fetch_ohlc_coingecko(coin_id)
    df_raw = df_metrics.merge(df_ohlc, on="date", how="inner").sort_values("date")
    save_csv(df_raw, f"{coin}_raw.csv")

    # Step 2: Load raw + Feature engineering
    df_raw = load_csv(f"{coin}_raw.csv")
    base_cols = list(BITINFO_METRICS.keys()) + ["open", "high", "low", "close"]
    df_feat = add_indicators(df_raw, base_cols)
    save_csv(df_feat, f"{coin}_features.csv")

    # Step 3: Load features + intervals + model
    df_feat = load_csv(f"{coin}_features.csv")

    results = []
    for interval_name, (start, end) in INTERVALS.items():
        df_interval = df_feat[(df_feat["date"] >= start) & (df_feat["date"] <= end)].copy()
        save_csv(df_interval, f"{coin}_{interval_name}.csv")

        for horizon in HORIZONS:
            target = f"close_t+{horizon}"
            df_interval[target] = df_interval["close"].shift(-horizon)
            data = df_interval.dropna().copy()

            # Save horizon dataset
            save_csv(data, f"{coin}_{interval_name}_h{horizon}_dataset.csv")

            X_train, X_test, y_train, y_test, feature_names = preprocess_split(data, target)
            keep_idx = select_features_mdi(X_train, y_train, feature_names)
            X_train_sel = X_train[:, keep_idx]
            X_test_sel = X_test[:, keep_idx]

            X_train_3d, y_train_3d = make_sliding_window(X_train_sel, y_train, window=1)
            X_test_3d, y_test_3d = make_sliding_window(X_test_sel, y_test, window=1)

            model = build_bilstm((X_train_3d.shape[1], X_train_3d.shape[2]))
            model.fit(X_train_3d, y_train_3d, epochs=10, batch_size=32, verbose=0)
            preds = model.predict(X_test_3d).flatten()

            # Save predictions CSV
            pred_df = pd.DataFrame({"y_true": y_test_3d, "y_pred": preds})
            save_csv(pred_df, f"{coin}_{interval_name}_h{horizon}_predictions.csv")

            rmse, mae, mape, r2 = report_metrics(y_test_3d, preds)
            results.append([interval_name, horizon, rmse, mae, mape, r2])
            print(f"{coin} | {interval_name} | horizon {horizon}d -> RMSE={rmse:.4f}, MAE={mae:.4f}, MAPE={mape:.2f}%, R2={r2:.4f}")

    results_df = pd.DataFrame(results, columns=["interval", "horizon_days", "RMSE", "MAE", "MAPE", "R2"])
    save_csv(results_df, f"{coin}_results.csv")
    print("\nSummary results:")
    print(results_df.head())

    # Step 4: CUSUM out-of-sample
    start, end = CUSUM_TEST
    df_cusum = df_feat[(df_feat["date"] >= start) & (df_feat["date"] <= end)].copy()
    df_cusum["target"] = df_cusum["close"].shift(-1)
    data = df_cusum.dropna().copy()

    X_train, X_test, y_train, y_test, feature_names = preprocess_split(data, "target")
    keep_idx = select_features_mdi(X_train, y_train, feature_names)
    X_train_sel = X_train[:, keep_idx]
    X_test_sel = X_test[:, keep_idx]

    X_train_3d, y_train_3d = make_sliding_window(X_train_sel, y_train, window=1)
    X_test_3d, y_test_3d = make_sliding_window(X_test_sel, y_test, window=1)

    model = build_bilstm((X_train_3d.shape[1], X_train_3d.shape[2]))
    model.fit(X_train_3d, y_train_3d, epochs=10, batch_size=32, verbose=0)
    preds = model.predict(X_test_3d).flatten()

    CUSUM_Control_Chart(preds, y_test_3d, model, X_train_3d, y_train_3d)

In [ ]:
# Run for BTC and ETH
# run_pipeline("btc", "bitcoin")
# run_pipeline("eth", "ethereum")